In [ ]:
!pip install docling langchain langchain-huggingface langchain-core langchain-groq

In [ ]:
import os
import base64
from pathlib import Path
from io import BytesIO
import pandas as pd
from PIL import Image
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat, DocumentStream
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional
from huggingface_hub import InferenceClient
from google.colab import userdata

# ── 1. API keys ──────────────────────────────
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HF_TOKEN')

# ── 2. Invoice schema ──────────────────────────────
class LineItem(BaseModel):
    quantity: int
    description: str
    unit_price: float
    total: float

class Invoice(BaseModel):
    invoice_number: str
    date: str = Field(description="Invoice date in YYYY-MM-DD format, converted from whatever format appears on the original document")
    vendor_name: str
    vendor_address: Optional[str] = None
    vendor_phone: Optional[str] = None
    bill_to_name: str
    bill_to_address: Optional[str] = None
    ship_to_name: Optional[str] = None
    ship_to_address: Optional[str] = None
    salesperson: Optional[str] = None
    po_number: Optional[str] = None
    terms: Optional[str] = None
    line_items: List[LineItem]
    subtotal: float
    tax: float
    shipping: Optional[float] = None
    total_due: float

# ── 3. VLM verification result schema ──────────────────────────────
class VLMVerificationResult(BaseModel):
    matches: bool = Field(description="True if the extracted data matches the image, False if there are mismatches")
    issues: List[str] = Field(default_factory=list, description="Specific mismatches found, empty list if none")

# ── 4. Docling converter, configured to generate page images (for VLM step) ──────────────────────────────
pipeline_options = PdfPipelineOptions()
pipeline_options.generate_page_images = True
pipeline_options.images_scale = 2.0

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# ── 5. LLM + prompt + parser for invoice extraction ──────────────────────────────
chat_model = ChatGroq(
    model="openai/gpt-oss-120b",
    max_tokens=4096,
    temperature=0
)

parser = PydanticOutputParser(pydantic_object=Invoice)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert at extracting structured data from invoice documents. "
                "Always output the date in YYYY-MM-DD format, converting from whatever format appears on the original document. "
                "Some line items may be split across a page break — if you see an orphaned product code or partial line with no matching description, merge it with the line item immediately before it. "
                "Extract all fields accurately from the markdown below.\n\n{format_instructions}"),
    ("human", "{invoice_markdown}")
])

chain = prompt | chat_model | parser

# ── 6. VLM parser + verification function ──────────────────────────────
vlm_parser = PydanticOutputParser(pydantic_object=VLMVerificationResult)
VLM_MODEL_CANDIDATES = ["zai-org/GLM-4.5V", "Qwen/Qwen2.5-VL-3B-Instruct"]

def verify_with_vlm(image_b64: str, extracted_invoice: Invoice) -> VLMVerificationResult:
    client = InferenceClient(api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"])
    extracted_json = extracted_invoice.model_dump_json(indent=2)

    prompt_text = (
        f"Here is data extracted from this invoice image:\n\n{extracted_json}\n\n"
        f"Carefully compare it against the image and check for mismatches.\n\n"
        f"{vlm_parser.get_format_instructions()}"
    )

    for model_name in VLM_MODEL_CANDIDATES:
        try:
            completion = client.chat.completions.create(
                model=model_name,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_text},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}}
                    ]
                }],
                max_tokens=1024,
            )
            raw_response = completion.choices[0].message.content
            return vlm_parser.parse(raw_response)
        except Exception:
            continue

    return None  # every candidate failed — treated as "Not checked"

# ── 7. Invoice validation function ──────────────────────────────
def validate_invoice(invoice: Invoice, tolerance: float = 0.01) -> dict:
    errors = []
    warnings = []

    calculated_subtotal = sum(item.total for item in invoice.line_items)
    if abs(calculated_subtotal - invoice.subtotal) > tolerance:
        errors.append(f"Line items sum to {calculated_subtotal}, but subtotal says {invoice.subtotal}")

    shipping = invoice.shipping or 0
    calculated_total = invoice.subtotal + invoice.tax + shipping
    if abs(calculated_total - invoice.total_due) > tolerance:
        errors.append(f"Subtotal+tax+shipping = {calculated_total}, but total_due says {invoice.total_due}")

    for i, item in enumerate(invoice.line_items):
        expected = item.quantity * item.unit_price
        if abs(expected - item.total) > tolerance:
            warnings.append(f"Line item {i} ('{item.description}'): {item.quantity} x {item.unit_price} = {expected}, but total says {item.total}")

    required_fields = {
        "invoice_number": invoice.invoice_number,
        "total_due": invoice.total_due,
        "vendor_name": invoice.vendor_name,
    }
    for field_name, value in required_fields.items():
        if not value:
            errors.append(f"Missing required field: {field_name}")

    if invoice.total_due <= 0:
        errors.append("total_due is zero or negative — likely extraction failure")
    for i, item in enumerate(invoice.line_items):
        if item.quantity <= 0:
            warnings.append(f"Line item {i} has non-positive quantity: {item.quantity}")

    return {
        "is_valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
    }

# ── 8. In-memory .jfif handling — no disk writes, single folder ──────────────────────────────
def get_docling_input(filepath: Path):
    """
    Returns what Docling should convert.
    .jfif files are converted to JPEG entirely in memory — nothing is written to disk,
    so the source folder is never modified while being processed.
    Everything else is returned as-is (the original Path).
    """
    if filepath.suffix.lower() == ".jfif":
        img = Image.open(filepath).convert("RGB")
        buffer = BytesIO()
        img.save(buffer, format="JPEG")
        buffer.seek(0)
        return DocumentStream(name=filepath.stem + ".jpg", stream=buffer)
    return filepath

def image_to_base64(pil_image: Image.Image) -> str:
    buffer = BytesIO()
    pil_image.convert("RGB").save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

# ── 9. Row-building functions ──────────────────────────────
def invoice_to_summary_row(invoice: Invoice, validation_result: dict, vlm_result: VLMVerificationResult) -> dict:
    if vlm_result is None:
        vlm_status = "Not checked"
        vlm_issues_text = "None"
    elif vlm_result.matches:
        vlm_status = "Yes"
        vlm_issues_text = "None"
    else:
        vlm_status = "No"
        vlm_issues_text = "; ".join(vlm_result.issues) if vlm_result.issues else "None"

    return {
        "invoice_number": invoice.invoice_number,
        "date": invoice.date,
        "vendor_name": invoice.vendor_name,
        "bill_to_name": invoice.bill_to_name,
        "subtotal": invoice.subtotal,
        "tax": invoice.tax,
        "shipping": invoice.shipping if invoice.shipping is not None else 0,
        "total_due": invoice.total_due,
        "line_item_count": len(invoice.line_items),
        "status": "OK" if validation_result["is_valid"] else "NEEDS REVIEW",
        "errors": "; ".join(validation_result["errors"]) if validation_result["errors"] else "None",
        "warnings": "; ".join(validation_result["warnings"]) if validation_result["warnings"] else "None",
        "vlm_validated": vlm_status,
        "vlm_issues": vlm_issues_text,
    }

def invoice_to_line_item_rows(invoice: Invoice) -> list[dict]:
    rows = []
    for item in invoice.line_items:
        rows.append({
            "invoice_number": invoice.invoice_number,
            "description": item.description,
            "quantity": item.quantity,
            "unit_price": item.unit_price,
            "total": item.total,
        })
    return rows

# ── 10. Batch process every supported file in ONE folder ──────────────────────────────
INVOICE_FOLDER = Path("/content/drive/MyDrive/invoices")  # update to your actual path
MIN_MARKDOWN_LENGTH = 50

all_summary_rows = []
all_line_item_rows = []
failed_files = []

# Snapshot the file list ONCE before processing — nothing created during the loop can be picked up
files_to_process = [
    f for f in INVOICE_FOLDER.iterdir()
    if f.suffix.lower() in [".pdf", ".jpg", ".jpeg", ".png", ".webp", ".jfif"]
]

for filepath in files_to_process:
    print(f"Processing: {filepath.name}")

    try:
        docling_input = get_docling_input(filepath)

        docling_result = converter.convert(docling_input)
        markdown_text = docling_result.document.export_to_markdown()

        # Defense layer 1: catch OCR failures before calling the LLM
        if len(markdown_text.strip()) < MIN_MARKDOWN_LENGTH:
            print(f"  FAILED: OCR extracted almost no text ({len(markdown_text.strip())} chars) — skipping LLM call")
            failed_files.append({
                "filename": filepath.name,
                "error": f"OCR extraction produced insufficient text ({len(markdown_text.strip())} chars) — likely OCR failure"
            })
            continue

        # Structure with LLM
        result = chain.invoke({
            "invoice_markdown": markdown_text,
            "format_instructions": parser.get_format_instructions()
        })

        # Validate (math/required fields)
        validation_result = validate_invoice(result)

        # Defense layer 2: VLM cross-check against the actual page image
        vlm_result = None
        try:
            if filepath.suffix.lower() == ".pdf":
                first_page = next(iter(docling_result.document.pages.values()))
                page_image = first_page.image.pil_image
            else:
                # Works for .jfif too — reopened directly from the original file, nothing saved to disk
                page_image = Image.open(filepath).convert("RGB")

            image_b64 = image_to_base64(page_image)
            vlm_result = verify_with_vlm(image_b64, result)
        except Exception as vlm_error:
            print(f"  NOTE: VLM verification unavailable for this file: {vlm_error}")

        # Print issues found
        for e in validation_result["errors"]:
            print(f"  ERROR: {e}")
        for w in validation_result["warnings"]:
            print(f"  WARNING: {w}")
        if vlm_result is not None:
            print(f"  VLM validated: {vlm_result.matches}")

        # Build and collect rows
        summary_row = invoice_to_summary_row(result, validation_result, vlm_result)
        summary_row["source_file"] = filepath.name
        all_summary_rows.append(summary_row)

        line_item_rows = invoice_to_line_item_rows(result)
        all_line_item_rows.extend(line_item_rows)

    except Exception as e:
        print(f"  FAILED to process {filepath.name}: {e}")
        failed_files.append({"filename": filepath.name, "error": str(e)})

# ── 11. Save everything once, at the end ──────────────────────────────
pd.DataFrame(all_summary_rows).to_csv("invoices_summary.csv", index=False)
pd.DataFrame(all_line_item_rows).to_csv("line_items_detail.csv", index=False)

if failed_files:
    pd.DataFrame(failed_files).to_csv("failed_files.csv", index=False)

print(f"\nDone. Processed {len(all_summary_rows)} invoices successfully, {len(failed_files)} failed.")